# Notebook 2 — Keyword & Filtered Search

*Hands-on time: ~30 minutes*

Now that data is indexed, we explore ElasticSearch's **structured query** capabilities:
1. Match queries (full-text / BM25)
2. Term queries (exact keyword match)
3. Range filters (numeric fields)
4. Bool compound queries (`must` / `should` / `filter`)
5. Aggregations (faceted analytics)

**Prerequisites:** Complete Notebook 1.

In [ ]:
from elasticsearch import Elasticsearch
import pandas as pd

client = Elasticsearch("http://localhost:9200")
INDEX_NAME = "neuroimaging"

count = client.count(index=INDEX_NAME)
print(f"Index '{INDEX_NAME}' has {count['count']} documents")


def show_hits(response, fields=None):
    """Pretty-print search results as a DataFrame."""
    rows = []
    for hit in response["hits"]["hits"]:
        row = {"_score": hit["_score"]}
        src = hit["_source"]
        if fields:
            for f in fields:
                row[f] = src.get(f)
        else:
            # Exclude embedding for readability
            row.update({k: v for k, v in src.items() if k != "metadata_embedding"})
        rows.append(row)
    return pd.DataFrame(rows)

---
## 1. Match Queries — Full-Text Search (BM25)

A `match` query analyzes the input text and uses BM25 scoring against
analyzed `text` fields like `description_text` and `SeriesDescription`.

In [ ]:
# Full-text search over the description_text field
response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "match": {
                "description_text": "anatomical structural T1"
            }
        },
        "size": 5
    }
)

print(f"Hits: {response['hits']['total']['value']}")
show_hits(response, ["suffix", "subject", "task", "description_text", "bids_path"])

In [ ]:
# Multi-match: search across multiple text fields at once
response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "multi_match": {
                "query": "balloon analog risk",
                "fields": ["TaskName", "description_text"],
                "type": "best_fields"
            }
        },
        "size": 5
    }
)

print(f"Hits: {response['hits']['total']['value']}")
show_hits(response, ["suffix", "subject", "TaskName", "bids_path"])

---
## 2. Term Queries — Exact Match on Keywords

For `keyword` fields, use `term` (single value) or `terms` (multiple values).
These are **exact** matches — no text analysis.

In [ ]:
# Find all BOLD scans
response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "term": {"suffix": "bold"}
        },
        "size": 5
    }
)

print(f"BOLD hits: {response['hits']['total']['value']}")
show_hits(response, ["subject", "task", "run", "RepetitionTime", "bids_path"])

In [ ]:
# Find scans from a specific subject
response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "term": {"subject": "01"}
        },
        "size": 20
    }
)

print(f"Subject 01 scans: {response['hits']['total']['value']}")
show_hits(response, ["subject", "suffix", "task", "run", "bids_path"])

In [ ]:
# Match multiple values: T1w OR bold
response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "terms": {"suffix": ["T1w", "bold"]}
        },
        "size": 5
    }
)

print(f"T1w or BOLD hits: {response['hits']['total']['value']}")
show_hits(response, ["subject", "suffix", "datatype", "bids_path"])

---
## 3. Range Filters — Numeric Fields

Range queries let you filter by numeric values: TR, TE, field strength, etc.

In [ ]:
# Find scans with RepetitionTime > 1.5 seconds
response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "range": {
                "RepetitionTime": {"gt": 1.5}
            }
        },
        "size": 5
    }
)

print(f"TR > 1.5s: {response['hits']['total']['value']} hits")
show_hits(response, ["subject", "suffix", "task", "RepetitionTime", "bids_path"])

In [ ]:
# Range with bounds: 0.8s ≤ TR ≤ 2.0s
response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "range": {
                "RepetitionTime": {"gte": 0.8, "lte": 2.0}
            }
        },
        "size": 5
    }
)

print(f"0.8s ≤ TR ≤ 2.0s: {response['hits']['total']['value']} hits")
show_hits(response, ["subject", "suffix", "RepetitionTime", "bids_path"])

---
## 4. Bool Compound Queries

The `bool` query combines clauses:
- **must**: All clauses must match (AND) — affects score
- **should**: At least one should match (OR) — boosts score
- **filter**: Must match — does NOT affect score (faster, cached)
- **must_not**: Must not match (exclusion)

In [ ]:
# Complex query:
#   - MUST be a BOLD scan
#   - FILTER by TR between 1.0 and 3.0
#   - SHOULD match "balloon" in the task name (boosts score if present)
response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "bool": {
                "must": [
                    {"term": {"suffix": "bold"}}
                ],
                "filter": [
                    {"range": {"RepetitionTime": {"gte": 1.0, "lte": 3.0}}}
                ],
                "should": [
                    {"match": {"TaskName": "balloon"}}
                ]
            }
        },
        "size": 10
    }
)

print(f"Compound query hits: {response['hits']['total']['value']}")
show_hits(response, ["subject", "task", "run", "RepetitionTime", "TaskName", "_score"])

In [ ]:
# Exclusion: all scans EXCEPT T1w
response = client.search(
    index=INDEX_NAME,
    body={
        "query": {
            "bool": {
                "must_not": [
                    {"term": {"suffix": "T1w"}}
                ]
            }
        },
        "size": 5
    }
)

print(f"Non-T1w scans: {response['hits']['total']['value']}")
show_hits(response, ["subject", "suffix", "task", "bids_path"])

---
## 5. Aggregations — Analytics & Faceted Counts

Aggregations let you compute statistics over your indexed data,
similar to SQL `GROUP BY`.

In [ ]:
# Count documents by suffix (scan type)
response = client.search(
    index=INDEX_NAME,
    body={
        "size": 0,
        "aggs": {
            "scan_types": {
                "terms": {"field": "suffix"}
            }
        }
    }
)

print("Scan types distribution:")
for bucket in response["aggregations"]["scan_types"]["buckets"]:
    print(f"  {bucket['key']:15s} {bucket['doc_count']:4d}")

In [ ]:
# Stats on RepetitionTime for BOLD scans only
response = client.search(
    index=INDEX_NAME,
    body={
        "size": 0,
        "query": {
            "term": {"suffix": "bold"}
        },
        "aggs": {
            "tr_stats": {
                "stats": {"field": "RepetitionTime"}
            }
        }
    }
)

stats = response["aggregations"]["tr_stats"]
print("RepetitionTime stats (BOLD scans):")
for k, v in stats.items():
    print(f"  {k}: {v}")

In [ ]:
# Nested aggregation: count by suffix, then sub-aggregate subjects per suffix
response = client.search(
    index=INDEX_NAME,
    body={
        "size": 0,
        "aggs": {
            "by_suffix": {
                "terms": {"field": "suffix"},
                "aggs": {
                    "unique_subjects": {
                        "cardinality": {"field": "subject"}
                    }
                }
            }
        }
    }
)

print("Scans by type with unique subject count:")
print(f"  {'Suffix':15s} {'Docs':>6s} {'Subjects':>10s}")
print(f"  {'-'*33}")
for bucket in response["aggregations"]["by_suffix"]["buckets"]:
    print(f"  {bucket['key']:15s} {bucket['doc_count']:6d} {bucket['unique_subjects']['value']:10.0f}")

In [ ]:
# Histogram aggregation: distribution of RepetitionTime
response = client.search(
    index=INDEX_NAME,
    body={
        "size": 0,
        "aggs": {
            "tr_histogram": {
                "histogram": {
                    "field": "RepetitionTime",
                    "interval": 0.5
                }
            }
        }
    }
)

print("RepetitionTime distribution (0.5s bins):")
for bucket in response["aggregations"]["tr_histogram"]["buckets"]:
    bar = "█" * bucket["doc_count"]
    print(f"  {bucket['key']:5.1f}s: {bar} ({bucket['doc_count']})")

---
## Exercise

Try writing your own queries:

1. Find all scans from subjects "03" OR "07"
2. Find BOLD scans with TR < 2.0s that mention "risk" in the TaskName
3. Compute the average FlipAngle across all documents that have one

*(Solutions hidden below — scroll down after trying!)*

In [ ]:
# YOUR CODE HERE — Exercise 1

In [ ]:
# YOUR CODE HERE — Exercise 2

In [ ]:
# YOUR CODE HERE — Exercise 3

<details>
<summary><b>Click for solutions</b></summary>

```python
# Exercise 1
client.search(index=INDEX_NAME, body={
    "query": {"terms": {"subject": ["03", "07"]}},
    "size": 20
})

# Exercise 2
client.search(index=INDEX_NAME, body={
    "query": {
        "bool": {
            "must": [{"term": {"suffix": "bold"}}, {"match": {"TaskName": "risk"}}],
            "filter": [{"range": {"RepetitionTime": {"lt": 2.0}}}]
        }
    },
    "size": 10
})

# Exercise 3
client.search(index=INDEX_NAME, body={
    "size": 0,
    "query": {"exists": {"field": "FlipAngle"}},
    "aggs": {"avg_flip_angle": {"avg": {"field": "FlipAngle"}}}
})
```
</details>

---
## Summary

You've learned:
- ✅ `match` and `multi_match` for full-text BM25 search
- ✅ `term` and `terms` for exact keyword matching
- ✅ `range` for numeric filtering
- ✅ `bool` queries to combine clauses with must/should/filter/must_not
- ✅ Aggregations: `terms`, `stats`, `cardinality`, `histogram`

**Next:** Open **Notebook 3** to unlock the power of **vector (semantic) search**.